# Parallel vs Sequential Chunk Execution

This notebook compares the two chunk-processing strategies available in `coffea-workflow`.

## What `parallel_chunks` controls

When you set `strategy="by_dataset"` and `percentage=20`, the workflow splits your fileset into **chunks** — one group of files per dataset per batch. You get a chunk per 20% of files, so 5 chunks per dataset.

Both strategies below use the same `DaskExecutor` backed by the coffea-casa cluster at `tls://localhost:8786`. The only difference is `parallel_chunks`.

**Sequential** (`parallel_chunks=False`, default): chunks are processed one after another in the main process. The `DaskExecutor` parallelises ROOT file reading *within* a chunk across Dask workers, but the next chunk only starts when the current one finishes.

**Parallel** (`parallel_chunks=True`): the outer chunk loop is also distributed. All uncached chunks are submitted to Dask workers at once via `client.submit()`. Each worker serialises the builder function from bytes (cloudpickle) and runs it with a local `IterativeExecutor`. Results are gathered and merged back in the main process.

```
Sequential  (parallel_chunks=False)
────────────────────────────────────────────────────────
  Chunk 0  [files ──► DaskExecutor (workers) ──► result_0]
  Chunk 1  [files ──► DaskExecutor (workers) ──► result_1]
  Chunk 2  [files ──► DaskExecutor (workers) ──► result_2]
  ...      (one chunk at a time; files within each chunk go to Dask workers)
  merge(result_0, result_1, result_2, ...)

Parallel  (parallel_chunks=True)
────────────────────────────────────────────────────────
  ┌─ Chunk 0  [files ──► IterativeExecutor ──► result_0] ─┐
  ├─ Chunk 1  [files ──► IterativeExecutor ──► result_1] ─┤  (all at once on Dask workers)
  └─ Chunk 2  [files ──► IterativeExecutor ──► result_2] ─┘
  client.gather(all futures)
  merge(result_0, result_1, result_2, ...)
```

**Constraints for `parallel_chunks=True`:**
- Requires `executor_type="DaskExecutor"` (needs a live Dask client)
- Incompatible with `hist_client` (gRPC connections cannot be pickled for workers)
- `coffea-workflow` does not need to be installed on workers — the builder function is serialised with `cloudpickle`

In [1]:
import sys
import time

sys.path.insert(0, "..")

from coffea_workflow import Step, Workflow, Fileset, Analysis, Plotting, RunConfig, ExecutorConfig, run
from coffea_workflow import facilities
from analysis import get_fileset, run_analysis, plot_results

In [2]:
def build_workflow():
    """Returns a fresh Workflow with the standard fileset → analysis → plotting chain."""
    step_fileset = Step(
        name="Fileset",
        step_type=Fileset,
        builder=get_fileset,
        builder_params={"with_failure": True},
        output="fileset_dict",
    )
    step_analysis = Step(
        name="Analysis",
        step_type=Analysis,
        builder=run_analysis,
        input="fileset_dict",
        output="analysis_payload",
    )
    step_plotting = Step(
        name="Plotting",
        step_type=Plotting,
        builder=plot_results,
        input="analysis_payload",
    )
    wf = Workflow()
    wf.add(step_fileset)
    wf.add(step_analysis, depends_on=[step_fileset])
    wf.add(step_plotting, depends_on=[step_analysis])
    return wf

In [3]:
# Shared facility — connects to the coffea-casa Dask cluster and installs
# the required packages on every worker. Reused by both configs below.
facility = facilities.CoffeaCasaFactory(
    worker_packages=("git+https://github.com/hooloobooroodkoo/coffea.git@processor_result_type",),
    worker_files=("../analysis.py",),
)

## Strategy 1: Sequential (`parallel_chunks=False`)

Chunks are processed one after another in the main process. For each chunk, coffea's `DaskExecutor` is passed to `run_analysis` — Dask workers handle the ROOT file reading and event processing in parallel *within* that chunk. The next chunk only starts after the current one completes.

In [3]:
config_sequential = RunConfig(
    strategy="by_dataset",
    percentage=20,          # 5 chunks per dataset, 10 chunks total
    cache_dir=".cache_sequential_with_failure",
    facility=facility,
    executor_config=ExecutorConfig(
        executor_type="DaskExecutor",   # coffea uses DaskExecutor inside each chunk
        parallel_chunks=False,          # chunks processed one at a time (default)
    ),
)

t0 = time.perf_counter()
result_sequential = run(build_workflow(), config_sequential)
t_sequential = time.perf_counter() - t0

print(f"\nSequential wall time: {t_sequential:.1f}s")

Workflow DAG:
  [0] Fileset -> Fileset builder=<function get_fileset at 0x7f29b130bf60>
  [1] Analysis -> Analysis builder=<function run_analysis at 0x7f29b12f1080>
  [2] Plotting -> Plotting builder=<function plot_results at 0x7f29b12f13a0>
Edges:
  Fileset -> Analysis
  Analysis -> Plotting
Run config:
  Strategy:  by_dataset
  Executor:  DaskExecutor  workers=1
  Facility:  CoffeaCasaFactory
Executing step 'Fileset' of type 'Fileset' with the user code <function get_fileset at 0x7f29b130bf60> and user parameters {'with_failure': True}
Extracted from cache: .cache_sequential_with_failure/Fileset/c016b39da53c1aad6ab919396f49a19da7473f593f3e9bc9a9efa65b3fb27f4b
  -> materialized at .cache_sequential_with_failure/Fileset/c016b39da53c1aad6ab919396f49a19da7473f593f3e9bc9a9efa65b3fb27f4b
Executing step 'Analysis' of type 'Analysis' with the user code <function run_analysis at 0x7f29b12f1080> and user parameters None
Extracted from cache: .cache_sequential_with_failure/Chunking/f2fb1933f638

Output()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /usr/local/lib/python3.12/site-packages/distributed/worker.py:2982 in _run_task_simple           │
│                                                                                                  │
│   2979 │   │   context_meter.meter("thread-cpu", func=thread_time),                              │
│   2980 │   ):                                                                                    │
│   2981 │   │   try:                                                                              │
│ ❱ 2982 │   │   │   result = task(data)                                                           │
│   2983 │   │   except (SystemExit, KeyboardInterrupt):                                           │
│   2984 │   │   │   # Special-case these, just like asyncio does all over the place. They will    │
│   2985 │   │   │   # pass through `fail_hard` and `_handle_stimulus_from_task`, and eventually   │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/dask/_task_spec.py:755 in __call__                       │
│                                                                                                  │
│    752 │   │   │   │   for k, kw in self.kwargs.items()                                          │
│    753 │   │   │   }                                                                             │
│    754 │   │   │   return self.func(*new_argspec, **kwargs)                                      │
│ ❱  755 │   │   return self.func(*new_argspec)                                                    │
│    756 │                                                                                         │
│    757 │   def __setstate__(self, state):                                                        │
│    758 │   │   slots = self.__class__.get_all_slots()                                            │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:1290 in automatic_retries   │
│                                                                                                  │
│   1287 │   │   │   │   │   if use_result_type:                                                   │
│   1288 │   │   │   │   │   │   # surface the exception instead of silently skipping              │
│   1289 │   │   │   │   │   │   # so the Runner can wrap it as Err                                │
│ ❱ 1290 │   │   │   │   │   │   raise e                                                           │
│   1291 │   │   │   │   │   warnings.warn(                                                        │
│   1292 │   │   │   │   │   │   f"Skipping bad file after {retry_count + 1} attempts. The last e  │
│   1293 │   │   │   │   │   )                                                                     │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:1276 in automatic_retries   │
│                                                                                                  │
│   1273 │   │   retry_count = 0                                                                   │
│   1274 │   │   while retry_count <= retries:                                                     │
│   1275 │   │   │   try:                                                                          │
│ ❱ 1276 │   │   │   │   return func(*args, **kwargs)                                              │
│   1277 │   │   │   except Exception as e:                                                        │
│   1278 │   │   │   │   warnings.warn(                                                            │
│   1279 │   │   │   │   │   f"Performed attempt {retry_count

Failure caught!
------------------------------------
Processing fileset_chunk_1.json
Extracted from cache: .cache_sequential_with_failure/ChunkAnalysis/828b42b4a82b6117f8ef44a4fc1095a8da17eb420841eb85c5550765acd2b9d6
Successfully processed!
------------------------------------
Processing fileset_chunk_2.json
Extracted from cache: .cache_sequential_with_failure/ChunkAnalysis/62800d4d5f08853d40d2aa3c38168f39afed79f0659358f8a2568c3abca555d8
Successfully processed!
------------------------------------
Processing fileset_chunk_3.json
Extracted from cache: .cache_sequential_with_failure/ChunkAnalysis/ba947731e025243f042c969939448c4a34dfc7cbcd503fbcaad3e1eb40aa7f27
Successfully processed!
------------------------------------
Processing fileset_chunk_4.json
Extracted from cache: .cache_sequential_with_failure/ChunkAnalysis/802a6e01627b6a308989046c4e8629f837d4b5559fdcf5237ffe4f283d31184a
Successfully processed!
------------------------------------
Processing fileset_chunk_5.json
Extracted from 

Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26.7s
Sequential wall time: 26

RecursionError: maximum recursion depth exceeded

## Strategy 2: Parallel chunks (`parallel_chunks=True`)

All uncached chunks are submitted to Dask workers at once via `client.submit()`. On each worker, `coffea-workflow` overrides the executor and runs the builder with a local `IterativeExecutor` — so there is no nested Dask parallelism inside a chunk. The chunk-level parallelism comes from all chunks running simultaneously across workers.

`executor_type="DaskExecutor"` is still required here — not for within-chunk file reading, but because `parallel_chunks=True` needs `coffea_exec.client` to submit futures. The builder function is serialised with `cloudpickle`, so `coffea-workflow` does not need to be installed on the workers.

In [ ]:
config_parallel = RunConfig(
    strategy="by_dataset",
    percentage=20,          # same 10 chunks — all submitted to workers at once
    cache_dir=".cache_parallel_with_failure",
    facility=facility,
    executor_config=ExecutorConfig(
        executor_type="DaskExecutor",   # required to provide client for chunk submission;
                                        # workers themselves run IterativeExecutor per chunk
        parallel_chunks=True,
    ),
)

t0 = time.perf_counter()
result_parallel = run(build_workflow(), config_parallel)
t_parallel = time.perf_counter() - t0

# print(f"\nParallel wall time: {t_parallel:.1f}s")